In [ ]:
import pandas as pd
from paths import DATA

# `import paths` resolves with no sys.path handling because the project is
# installed editable (`pip install -e .`) with src as the import root.

In [ ]:
accepted_df = pd.read_csv(
    DATA / "accepted_2007_to_2018q4.csv" / "accepted_2007_to_2018Q4.csv",
    low_memory=False,
)

# The raw exports unpacked into directories NAMED like files, hence the two hops.
#
# low_memory=False silences the DtypeWarning you saw: without it pandas infers
# dtypes chunk-by-chunk and can assign different types to the same column, which
# is how `id` ends up part int and part str.
#
# rejected_df is out of scope for v1 (§6 of the findings doc: no loan_status, so
# no ground truth, ever) and costs minutes to load. Left commented rather than
# deleted so the decision stays visible:
# rejected_df = pd.read_csv(
#     DATA / "rejected_2007_to_2018q4.csv" / "rejected_2007_to_2018Q4.csv",
#     low_memory=False,
# )

In [ ]:
accepted_df['issue_d'] = pd.to_datetime(accepted_df['issue_d'], format='%b-%Y')


In [ ]:
LABEL_MAP = {
    'Fully Paid':  0,
    'Charged Off': 1,
    'Default':     1,
}

RESOLVED = set(LABEL_MAP)

accepted_36_month_population = accepted_df[
    (accepted_df['term'].str.strip() == '36 months')
    & (accepted_df['loan_status'].isin(RESOLVED))
    & (accepted_df['issue_d'] >= '2012-01-01')
    & (accepted_df['issue_d'] <= '2015-12-31')
].copy()
accepted_36_month_population['default'] = (
    accepted_36_month_population['loan_status'].map(LABEL_MAP).astype('int8')
)


In [ ]:
accepted_36_month_population['default'].value_counts()

# accepted_36_month_population['loan_status'].value_counts()

# accepted_36_month_population.shape[0]

# lbl = accepted_36_month_population['default']
# assert lbl.notna().all(), "unmapped loan_status leaked through"
# assert set(lbl.unique()) <= {0, 1}
# print(f"n={len(lbl):,}  base rate={lbl.mean():.4%}")   # expect 589,488 / 14.03%


In [ ]:
accepted_36_month_population.drop(columns=['loan_status'], inplace=True)


In [ ]:
accepted_36_month_population.iloc[0].to_dict()

In [ ]:
accepted_36_month_population.to_csv(
    DATA / "accepted_36_month_population.csv",
    index=False,
)

# This is the handoff point to notebook 2, which makes the filename a contract
# between two notebooks rather than a detail of this one. When pipeline.py takes
# this over, the name belongs there - as the return value of the function that
# writes it - not back in paths.py.
#
# Note what dropping `loan_status` above costs you: the axis-1 leakage audit in
# Step 3b is "null rate by loan_status", and this file no longer has it. Group by
# `default` instead - it is the same information, 1:1 recoded:
#
#     df.groupby("default")["settlement_amount"].apply(lambda s: s.isna().mean())
#
# That works because the label is a faithful recode of the status. It would NOT
# work if you had collapsed several statuses into one class unevenly.